In [1]:
!pip -q install onnx onnxruntime onnxsim
!pip install -q mediapipe==0.10.35
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
import os, json, shutil, glob
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

ANN_PATH = "/kaggle/input/datasets/nikita041/bukva-dataset/annotations.tsv"
DATASET_DIR = Path("/kaggle/input/datasets/nikita041/bukva-dataset")
TRIMMED_DIR = DATASET_DIR / "trimmed"
ANN_PATH    = DATASET_DIR / "annotations.tsv"

FRAMES_ROOT = Path("/kaggle/working/bukva_frames")
CKPT_DIR    = Path("/kaggle/working")

EXTRACT_FRAMES = True  # Поставь False, если кадры уже извлечены (например, вынес в отдельный датасет)
K_FRAMES = 8     # Оптимальное количество кадров из одного видео (и для train, и для val)
GESTURE_MARGIN = 0.15  # Отступ от краев видео, чтобы не брать переходные движения
CROP_PAD = 0.3        # Отступ вокруг руки (30%)

IMG_SIZE = 224       # Стандартный вход для MobileNetV2
BATCH_SIZE = 96

base_options = mp_python.BaseOptions(model_asset_path="hand_landmarker.task")
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,   # обрабатываем отдельные картинки
    num_hands=2,                             # ищем только одну руку
    min_hand_detection_confidence=0.3,       # порог 0.3, чтобы ловить чуть смазанные руки
)
detector = vision.HandLandmarker.create_from_options(options)

# Словари классов (только полезные буквы + no_event, если нужно)
CLASSES = {
    0: "no_event", 1: "Ё", 2: "А", 3: "Б", 4: "В", 5: "Г", 6: "Д", 7: "Е",
    8: "Ж", 9: "З", 10: "И", 11: "Й", 12: "К", 13: "Л", 14: "М", 15: "Н",
    16: "О", 17: "П", 18: "Р", 19: "С", 20: "Т", 21: "У", 22: "Ф", 23: "Х",
    24: "Ц", 25: "Ч", 26: "Ш", 27: "Щ", 28: "Ъ", 29: "Ы", 30: "Ь", 31: "Э",
    32: "Ю", 33: "Я",
}
LETTER2ID = {v: k for k, v in CLASSES.items()}

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1784096600.628779      84 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784096600.643341      84 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
def get_hand_crop(frame):
    h, w = frame.shape[ :2]
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    detection_result = detector.detect(mp_image)
    
    if not detection_result.hand_landmarks:
        s = min(h, w)
        y0, x0 = (h - s) // 2, (w - s) // 2
        crop = frame[y0:y0+s, x0:x0+s]
        return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

    # Выбор руки
    best_landmarks = None
    max_area = 0
    
    for landmarks in detection_result.hand_landmarks:
        x_coords = [lm.x for lm in landmarks]
        y_coords = [lm.y for lm in landmarks]
        area = (max(x_coords) - min(x_coords)) * (max(y_coords) - min(y_coords))
        if area > max_area:
            max_area = area
            best_landmarks = landmarks
    # Дальше используем только лучшую рукуа
    landmarks = best_landmarks
    # Вытаскиваем координаты (x, y) для всех 21 точек
    x_coords = [lm.x for lm in landmarks]
    y_coords = [lm.y for lm in landmarks]
    
    # Переводим относительные координаты (от 0 до 1) в реальные пиксели
    x_min, x_max = int(min(x_coords) * w), int(max(x_coords) * w)
    y_min, y_max = int(min(y_coords) * h), int(max(y_coords) * h)
    
    # Делаем рамку квадратной
    box_w = x_max - x_min
    box_h = y_max - y_min
    side = max(box_w, box_h)
    
    # Добавляем отступы (padding), чтобы не обрезать кончики пальцев
    side = int(side * (1 + CROP_PAD))
    
    # Находим центр кисти
    cx = x_min + box_w // 2
    cy = y_min + box_h // 2
    
    # Вычисляем финальные координаты для обрезки
    new_x1 = max(0, cx - side // 2)
    new_y1 = max(0, cy - side // 2)
    new_x2 = min(w, cx + side // 2)
    new_y2 = min(h, cy + side // 2)
    
    crop = frame[new_y1:new_y2, new_x1:new_x2]
    
    # Защита от ошибок на краях кадра (если side вышел за границы)
    if crop.size == 0:
        return cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

# Выбирает k равномерных индексов (номеров кадров) на отрезке [lo, hi]
def sample_indices(lo, hi, k):
    lo, hi = int(max(0, lo)), int(hi)
    if k <= 0 or hi < lo: return []
    if hi == lo: return [lo]
    step = (hi - lo) / float(k)
    return [int(round(lo + step * (i + 0.5))) for i in range(k)]

In [4]:
if FRAMES_ROOT.exists(): shutil.rmtree(FRAMES_ROOT)
ann = pd.read_csv(ANN_PATH, sep="\t")

for _, row in tqdm(ann.iterrows(), total=len(ann), desc="Умная нарезка с MediaPipe Tasks API"):
    aid = str(row["attachment_id"])
    letter = str(row["text"]).strip()
    cls_id = LETTER2ID.get(letter)
    if cls_id is None: continue
    
    hits = glob.glob(str(TRIMMED_DIR / f"{aid}*"))
    if not hits: continue
    
    cap = cv2.VideoCapture(hits[0])
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_frames <= 1: continue
    
    lo, hi = round(GESTURE_MARGIN * n_frames), n_frames - 1 - round(GESTURE_MARGIN * n_frames)
    target_idx = set(sample_indices(lo, max(lo, hi), K_FRAMES))
    
    split = "train" if bool(row["train"]) else "val"
    save_dir = FRAMES_ROOT / split / f"{cls_id:02d}"
    save_dir.mkdir(parents=True, exist_ok=True)
    
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or idx > max(target_idx): break
        if idx in target_idx:
            smart_crop = get_hand_crop(frame)
            out_path = str(save_dir / f"{aid}_{idx}.jpg")
            cv2.imwrite(out_path, smart_crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
        idx += 1
    cap.release()

Умная нарезка с MediaPipe Tasks API:   0%|          | 0/3862 [00:00<?, ?it/s]

W0000 00:00:1784096601.105236      85 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [5]:
ZIP_PATH = "/kaggle/working/bukva_frames_dataset"

print(f"Начинаю архивацию папки {FRAMES_ROOT}...")
shutil.make_archive(base_name=ZIP_PATH, format='zip', root_dir=FRAMES_ROOT)
print(f"Готово! Архив сохранен как: {ZIP_PATH}.zip")

Начинаю архивацию папки /kaggle/working/bukva_frames...
Готово! Архив сохранен как: /kaggle/working/bukva_frames_dataset.zip
